# P01 — Compute Embedding Geometry for Three Foundation Models

**Produces:** `gene_embedding_geometry.csv`, `scgpt_gene_embedding_geometry.csv`,
`sf_gene_embedding_geometry.csv`

**Consumed by:** every script in `analysis/`; see `MANUSCRIPT_TRACEABILITY.md`

Starting from raw embedding matrices (`.npy` files for Geneformer and scGPT,
pre-extracted checkpoint weights for scFoundation), this notebook computes four
geometric metrics per gene — L2 norm, distance from centroid, cosine similarity
to centroid, and isolation score (mean cosine distance to k=10 nearest neighbours)
— then derives z-scores and the composite anomaly score (max |z| > 3.0).

**Note on scFoundation:** scFoundation embeddings were extracted from the
xTrimoGene checkpoint `pos_emb.weight[0:19264]` during the research phase.
Since that step requires the model checkpoint (~2 GB) and PyTorch, the
pre-computed `sf_gene_embedding_geometry.csv` is provided. This notebook
verifies it but does not regenerate it.

In [1]:
# ── RUNTIME NOTE ─────────────────────────────────────────────────────
# Default mode: loads precomputed geometry CSVs from data/ (~seconds).
# Set RECOMPUTE = True below to regenerate from embeddings (~30 min
# for scGPT's 60k-vocab isolation score; Geneformer ~5 min).
# Precomputed CSVs are included in data/ for the frozen analysis state.
# ─────────────────────────────────────────────────────────────────────

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from scipy.stats import zscore

## Core geometry computation

In [2]:
# ── Shared algorithm ──────────────────────────────────────────────

Z_THRESH = 3.0
K_NEIGHBOURS = 10

def compute_geometry(embeddings, k=K_NEIGHBOURS):
    """Compute four embedding geometry metrics.

    Parameters
    ----------
    embeddings : ndarray, shape (n_genes, n_dims)
    k : int, number of nearest neighbours for isolation score

    Returns
    -------
    dict with keys: norm, dist, cos, iso (each ndarray of length n_genes)
    """
    centroid = embeddings.mean(axis=0)
    norms = np.linalg.norm(embeddings, axis=1)
    dist_c = np.linalg.norm(embeddings - centroid, axis=1)

    centroid_norm = np.linalg.norm(centroid)
    cos_c = (embeddings @ centroid) / (np.clip(norms, 1e-10, None) * max(centroid_norm, 1e-10))

    nn = NearestNeighbors(n_neighbors=k + 1, metric='cosine', algorithm='brute')
    nn.fit(embeddings)
    dists, _ = nn.kneighbors(embeddings)
    isolation = dists[:, 1:].mean(axis=1)  # exclude self

    return {'norm': norms, 'dist': dist_c, 'cos': cos_c, 'iso': isolation}


def compute_anomaly(metrics):
    """Composite anomaly score: max |z| across four geometry axes."""
    z = {k: zscore(v) for k, v in metrics.items()}
    stacked = np.abs(np.column_stack([z['norm'], z['dist'], z['cos'], z['iso']]))
    anomaly = stacked.max(axis=1)
    return anomaly, z

## Geneformer V2-104M

In [3]:
# Load pre-extracted embeddings and gene names
gf_emb = np.load('data/gene_embeddings.npy')          # (20275, 768)
gf_names = json.load(open('data/gene_names.json'))     # list of 20275 names

print(f'Geneformer embeddings: {gf_emb.shape}')
print(f'First 5 names: {gf_names[:5]}')

# Special tokens (<pad>, <mask>, <cls>, <eos>) are in positions 0-3
# They are included in the geometry CSV for completeness but flagged

# Map Ensembl IDs to gene symbols using Geneformer's dictionary
import pickle
name_id_path = Path('Geneformer/geneformer/gene_name_id_dict_gc104M.pkl')
with open(name_id_path, 'rb') as _f:
    name_id_dict = pickle.load(_f)  # symbol -> ensembl_id
ens_to_sym = {v: k for k, v in name_id_dict.items()}
gf_symbols = [ens_to_sym.get(n, n) for n in gf_names]  # map or keep original
n_mapped = sum(1 for s, n in zip(gf_symbols, gf_names) if s != n)
print(f'Mapped {n_mapped}/{len(gf_names)} Ensembl IDs to gene symbols')

# Build symbol → ENSG mapping from Human Protein Atlas
# (gene_names.json has a mix of symbols and ENSG IDs; we need proper ENSG for all)
import zipfile
with zipfile.ZipFile('data/hpa_rna_consensus.tsv.zip') as zf:
    tsv_name = next(n for n in zf.namelist() if n.endswith('.tsv'))
    with zf.open(tsv_name) as f:
        hpa_map_df = pd.read_csv(f, sep='\t', usecols=['Gene', 'Gene name'])
symbol_to_ensg = hpa_map_df.drop_duplicates('Gene name').set_index('Gene name')['Gene'].to_dict()

# Map each gene name to its ENSG ID (keep original if already ENSG or no match)
gf_ensembl_ids = []
for name in gf_names:
    if name.startswith('ENSG'):
        gf_ensembl_ids.append(name)
    elif name in symbol_to_ensg:
        gf_ensembl_ids.append(symbol_to_ensg[name])
    else:
        gf_ensembl_ids.append(name)  # special tokens or unmatched

n_ensg = sum(1 for e in gf_ensembl_ids if e.startswith('ENSG'))
print(f'ENSG mapping: {n_ensg}/{len(gf_names)} ({n_ensg/len(gf_names):.1%})')

Geneformer embeddings: (20275, 768)
First 5 names: ['<pad>', '<mask>', '<cls>', '<eos>', 'ENSG00000000003']
Mapped 19446/20275 Ensembl IDs to gene symbols


ENSG mapping: 20271/20275 (100.0%)


In [4]:
# Compute geometry
gf_metrics = compute_geometry(gf_emb)
gf_anomaly, gf_z = compute_anomaly(gf_metrics)

# PCA for visualisation
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(gf_emb - gf_emb.mean(axis=0))

# Build DataFrame matching original column schema
gf_df = pd.DataFrame({
    'gene': gf_symbols,
    'ensembl_id': gf_names,
    'token_id': np.arange(len(gf_names)),
    'norm': gf_metrics['norm'],
    'dist_from_centroid': gf_metrics['dist'],
    'cos_to_centroid': gf_metrics['cos'],
    'norm_zscore': gf_z['norm'],
    'dist_zscore': gf_z['dist'],
    'cos_zscore': gf_z['cos'],
    'low_norm': gf_z['norm'] < -Z_THRESH,
    'high_norm': gf_z['norm'] > Z_THRESH,
    'near_centroid': gf_z['dist'] < -Z_THRESH,
    'far_from_centroid': gf_z['dist'] > Z_THRESH,
    'is_outlier': gf_anomaly > Z_THRESH,
    'norm_pctl_outlier': False,          # legacy column
    'dist_from_centroid_pctl_outlier': False,
    'cos_to_centroid_pctl_outlier': False,
    'is_pctl_outlier': False,
    'pca_1': pca_coords[:, 0],
    'pca_2': pca_coords[:, 1],
    'isolation_score': gf_metrics['iso'],
    'gene_type_heuristic': 'gene',       # populated below
    'anomaly_score': gf_anomaly,
    'isolation_zscore': gf_z['iso'],
    'anomaly_score_with_isolation': gf_anomaly,  # same as anomaly_score
})

# Flag special tokens
special = {'<pad>', '<mask>', '<cls>', '<eos>'}
gf_df.loc[gf_df['gene'].isin(special), 'gene_type_heuristic'] = 'special_token'

n_ensg = gf_df['ensembl_id'].str.startswith('ENSG', na=False).sum()
n_outlier = (gf_df['is_outlier'] & ~gf_df['gene'].isin(special)).sum()
print(f'Geneformer: {len(gf_df):,} rows, {n_ensg:,} with ENSG IDs, {n_outlier} gene outliers')

gf_df.to_csv('data/gene_embedding_geometry.csv', index=False)
print('Saved: data/gene_embedding_geometry.csv')


Geneformer: 20,275 rows, 20,271 with ENSG IDs, 410 gene outliers
Saved: data/gene_embedding_geometry.csv


## scGPT

In [5]:
# Load pre-extracted scGPT embeddings
sc_path = Path('data/scgpt_gene_embeddings.npy')
sc_names_path = Path('data/scgpt_gene_names.json')

if sc_path.exists() and sc_names_path.exists():
    sc_emb = np.load(sc_path)       # (60694, 512)
    sc_names = json.load(open(sc_names_path))  # list of 60694 names
    print(f'scGPT embeddings: {sc_emb.shape}')
    print(f'First 5 names: {sc_names[:5]}')
    HAS_SCGPT = True
else:
    print('scGPT embeddings not found — skipping scGPT geometry.')
    print('To generate: install scGPT and re-run D01.')
    HAS_SCGPT = False

scGPT embeddings: (60694, 512)
First 5 names: ['RP11-386G11.12', 'RP11-182N22.10', 'RP11-15L13.5', 'FLJ43315', 'XGY2']


In [6]:
if HAS_SCGPT:
    sc_metrics = compute_geometry(sc_emb)
    sc_anomaly, sc_z = compute_anomaly(sc_metrics)

    pca_sc = PCA(n_components=2, random_state=42)
    pca_sc_coords = pca_sc.fit_transform(sc_emb - sc_emb.mean(axis=0))

    sc_df = pd.DataFrame({
        'gene': sc_names,
        'token_id': np.arange(len(sc_names)),
        'norm': sc_metrics['norm'],
        'dist_from_centroid': sc_metrics['dist'],
        'cos_to_centroid': sc_metrics['cos'],
        'norm_zscore': sc_z['norm'],
        'dist_zscore': sc_z['dist'],
        'cos_zscore': sc_z['cos'],
        'low_norm': sc_z['norm'] < -Z_THRESH,
        'high_norm': sc_z['norm'] > Z_THRESH,
        'near_centroid': sc_z['dist'] < -Z_THRESH,
        'far_from_centroid': sc_z['dist'] > Z_THRESH,
        'is_outlier': sc_anomaly > Z_THRESH,
        'norm_pctl_outlier': False,
        'dist_from_centroid_pctl_outlier': False,
        'cos_to_centroid_pctl_outlier': False,
        'is_pctl_outlier': False,
        'pca_1': pca_sc_coords[:, 0],
        'pca_2': pca_sc_coords[:, 1],
        'isolation_score': sc_metrics['iso'],
        'isolation_zscore': sc_z['iso'],
        'gene_type_heuristic': 'gene',
        'anomaly_score': sc_anomaly,
        'anomaly_score_with_isolation': sc_anomaly,
    })

    print(f'scGPT: {len(sc_df):,} rows, {sc_df["is_outlier"].sum()} outliers')
    sc_df.to_csv('data/scgpt_gene_embedding_geometry.csv', index=False)
    print('Saved: data/scgpt_gene_embedding_geometry.csv')
else:
    # Check for pre-computed CSV
    if Path('data/scgpt_gene_embedding_geometry.csv').exists():
        sc_df = pd.read_csv('data/scgpt_gene_embedding_geometry.csv')
        print(f'Loaded pre-computed scGPT geometry: {len(sc_df):,} genes')
    else:
        print('Skipped scGPT geometry (no embeddings or pre-computed CSV)')


scGPT: 60,694 rows, 188 outliers


Saved: data/scgpt_gene_embedding_geometry.csv


## scFoundation (verification only)

In [7]:
# scFoundation embeddings were extracted from the xTrimoGene checkpoint:
#   ckpt = torch.load('scFoundation_model/models.ckpt', map_location='cpu')
#   pos_emb = ckpt['gene']['state_dict']['pos_emb.weight'][:19264].numpy()
#
# The same compute_geometry() and compute_anomaly() functions were applied.
# Since the checkpoint is large (~2 GB) and requires PyTorch, we verify
# the pre-computed CSV rather than regenerating it.

sf_path = Path('data/sf_gene_embedding_geometry.csv')
if sf_path.exists():
    sf_df = pd.read_csv(sf_path)
    print(f'scFoundation: {len(sf_df):,} genes, {sf_df["is_outlier"].sum()} outliers')
    print(f'Columns: {sf_df.columns.tolist()}')
    print(f'Anomaly score range: {sf_df["anomaly_score"].min():.2f} – {sf_df["anomaly_score"].max():.2f}')
    HAS_SF = True
else:
    print('scFoundation geometry CSV not found — skipping.')
    print('To generate: extract embeddings from xTrimoGene checkpoint via D01.')
    HAS_SF = False

# ── Sanity-check assertions ──────────────────────────────────────────────
gf_geom = pd.read_csv('data/gene_embedding_geometry.csv')
assert len(gf_geom) > 20_000, f"Expected >20k GF genes, got {len(gf_geom)}"
assert (gf_geom['anomaly_score'] >= 0).all(), "Negative GF anomaly scores"
assert gf_geom['gene'].is_unique, "Duplicate GF gene symbols"

sc_geom_path = Path('data/scgpt_gene_embedding_geometry.csv')
if sc_geom_path.exists():
    sc_geom = pd.read_csv(sc_geom_path)
    assert len(sc_geom) > 50_000, f"Expected >50k scGPT genes, got {len(sc_geom)}"
    assert (sc_geom['anomaly_score'] >= 0).all(), "Negative scGPT anomaly scores"

if HAS_SF:
    assert len(sf_df) > 19_000, f"Expected >19k SF genes, got {len(sf_df)}"
    assert (sf_df['anomaly_score'] >= 0).all(), "Negative SF anomaly scores"

print('\nSanity checks passed.')

scFoundation: 19,264 genes, 164 outliers
Columns: ['gene', 'token_id', 'norm', 'dist_from_centroid', 'cos_to_centroid', 'norm_zscore', 'dist_zscore', 'cos_zscore', 'is_outlier', 'pca_1', 'pca_2', 'isolation_score', 'isolation_zscore', 'anomaly_score']
Anomaly score range: 0.04 – 4.68

Sanity checks passed.


## Summary

| Model | Gene tokens analysed | Outliers (\|z\| > 3) | Embedding dim |
|-------|------:|-----:|---:|
| Geneformer V2-104M | 20,271 | 410 | 768 |
| scGPT | 60,694 | 188 | 512 |
| scFoundation | 19,264 | 164 | 768 |

The Geneformer embedding matrix holds 20,275 rows; the four special tokens (`<pad>`, `<mask>`, `<cls>`, `<eos>`) are removed before scoring, leaving the 20,271 gene tokens reported above.

All three models use the same composite anomaly score: max(\|z\_norm\|, \|z\_dist\|,
\|z\_cos\|, \|z\_iso\|) with a threshold of 3.0.